# DegreeDetailExtract — v6 Training Notebook
### Optimized for Google Colab Free Tier (T4 GPU)

> **Session strategy:** This notebook is designed to be run across **multiple sessions**.
> Every epoch saves a full checkpoint to Google Drive. If your session disconnects,
> just re-run **Section 0 → Section 3 (skip Section 2 if dataset exists) → Section 5**.
> Training will resume from where it stopped.

**v6 improvements over v5:**
- Blur removed from augmentation (Donut's OCR degrades on blurry text)
- Ground-truth `issue_date` always normalized to `DD-MM-YYYY`
- `max_new_tokens=256` (was 128) — prevents XML tag truncation
- Per-field Exact Match & CER metrics printed each epoch
- Cosine LR schedule with warmup
- Label smoothing = 0.1
- Early stopping (patience=3) — saves GPU time
- Encoder unfreeze in final 3 epochs only
- Greedy decoding during training (fast), beam search only for final eval


## Section 0 — Environment Setup

> **Run this every session.** Takes ~3 min. Can safely re-run.


In [ ]:
# ── System fonts (minimal — avoids 300MB noto-cjk download) ─────────────
!apt-get install -y --fix-missing fonts-dejavu fonts-liberation -q 2>/dev/null

# ── Python packages ────────────────────────────────────────────────────────
# NOTE: We do NOT pin transformers/tokenizers/sentencepiece versions.
# Colab pre-installs these with matching pre-built wheels.
# Pinning them forces pip to rebuild tokenizers from Rust source → ERROR.
!pip install -q --upgrade transformers datasets sentencepiece

# These are NOT pre-installed in Colab, so install specific versions:
!pip install -q 'albumentations>=1.4.0,<2.0' opencv-python-headless
!pip install -q Faker Pillow tqdm rapidfuzz

print('Environment setup complete.')
# Verify transformers version
import transformers
print(f'transformers version: {transformers.__version__}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/DegreeDetailExtract'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted:', DRIVE_DIR)


In [ ]:
import os, sys

REPO = '/content/DegreeDetailExtract'
if not os.path.exists(REPO):
    !git clone https://github.com/Vrushti33/DegreeDetailExtract.git {REPO}
else:
    !git -C {REPO} pull --quiet

sys.path.insert(0, REPO)
os.chdir(REPO)
print(f'Repo ready at: {REPO}')


In [ ]:
# Download certificate fonts (with fallback to DejaVu if download fails)
import os, urllib.request

FONT_DIR = '/content/DegreeDetailExtract/cert_fonts'
os.makedirs(FONT_DIR, exist_ok=True)

FONTS = [
    ('CormorantGaramond-Regular.ttf',
     'https://github.com/googlefonts/Cormorant/raw/main/fonts/ttf/CormorantGaramond-Regular.ttf'),
    ('CormorantGaramond-Bold.ttf',
     'https://github.com/googlefonts/Cormorant/raw/main/fonts/ttf/CormorantGaramond-Bold.ttf'),
    ('CormorantGaramond-Italic.ttf',
     'https://github.com/googlefonts/Cormorant/raw/main/fonts/ttf/CormorantGaramond-Italic.ttf'),
    ('Cinzel-Regular.ttf',
     'https://fonts.gstatic.com/s/cinzel/v23/8vIU7ww63nmVQl4djHv0ECos.ttf'),
    ('IMFellEnglish-Regular.ttf',
     'https://fonts.gstatic.com/s/imfellenglish/v16/Ktk1ALSLW8zDe0rthJysWrnLsAz3F6mZVY9Y5w.ttf'),
    ('UnifrakturMaguntia-Book.ttf',
     'https://fonts.gstatic.com/s/unifrakturmaguntia/v22/WWXPlh98MazZmIvvLC382gaVKknnrkVZMnD55g.ttf'),
]

failed = []
for fname, url in FONTS:
    fpath = os.path.join(FONT_DIR, fname)
    if not os.path.exists(fpath):
        try:
            print(f'  Downloading {fname}...')
            urllib.request.urlretrieve(url, fpath)
        except Exception as e:
            print(f'  WARNING: Could not download {fname}: {e}')
            failed.append(fname)
    else:
        print(f'  Cached: {fname}')

if failed:
    print(f'\nSome fonts failed to download: {failed}')
    print('The generator will fall back to DejaVu fonts — rendering will still work.')
else:
    print('All fonts ready.')


## Section 1 — GPU Check

> Confirm you have a T4 before proceeding. If it says CPU, go to
> **Runtime > Change runtime type > T4 GPU** and restart.


In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB')
    if vram_gb < 14:
        print('WARNING: Less than 14GB VRAM — reduce BATCH_SIZE to 1 if OOM.')
else:
    print('WARNING: No GPU! Training will be extremely slow. Switch runtime type.')


## Section 2 — Generate v6 Dataset

> **SKIP THIS SECTION IF YOU ALREADY RAN IT ONCE.**
> The dataset is saved to Drive as a ZIP and reloaded automatically in Section 3.
> Only re-run if you want to regenerate from scratch.

**Free tier timing:** ~20-30 min for 3,000 certificates (25 per real cert).
The generated dataset is zipped to Drive so it survives session disconnects.


In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────
DATASET      = '/content/dataset_v6'
ZIP_PATH     = '/content/drive/MyDrive/DegreeDetailExtract/dataset_v6.zip'
REAL_CERTS   = '/content/DegreeDetailExtract/real_certs'

# 40 synthetic variations per real cert:
# 40 × ~60 real certs = ~2,400 total (~1,920 train, ~480 val).
# Optimal balance for Colab Free: takes ~20 min to generate and prevents overfitting
# when combined with on-the-fly augmentation in Section 4.
COUNT_PER_CERT = 40

FORCE_REGEN = False  # Set True ONLY to regenerate from scratch

import os, zipfile, shutil

if FORCE_REGEN:
    if os.path.exists(DATASET): shutil.rmtree(DATASET)
    if os.path.exists(ZIP_PATH): os.remove(ZIP_PATH)
    print('Force-regenerating dataset...')
else:
    print('FORCE_REGEN=False. Section 3 will restore from Drive cache.')
    print('Set FORCE_REGEN=True here to regenerate from scratch.')


In [ ]:
# Run generation (only if FORCE_REGEN=True above and no cached zip exists)
import os, zipfile, shutil

DATASET    = '/content/dataset_v6'
ZIP_PATH   = '/content/drive/MyDrive/DegreeDetailExtract/dataset_v6.zip'
REAL_CERTS = '/content/DegreeDetailExtract/real_certs'

if os.path.exists(ZIP_PATH) and not os.path.exists(DATASET):
    print('Restoring dataset from Drive...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Dataset restored.')
elif not os.path.exists(DATASET):
    print('No cached dataset found. Generating now (~25 min)...')
    os.makedirs(DATASET, exist_ok=True)
    !python /content/DegreeDetailExtract/generate_semi_synthetic_v6.py \
        --real_dir {REAL_CERTS} \
        --output_dir {DATASET} \
        --count_per_cert {COUNT_PER_CERT} \
        --seed 42
    print('Zipping to Drive...')
    shutil.make_archive('/content/dataset_v6', 'zip', '/content/', 'dataset_v6')
    shutil.copy('/content/dataset_v6.zip', ZIP_PATH)
    print('Saved to Drive:', ZIP_PATH)
else:
    print('Dataset already in /content — no action needed.')


## Section 3 — Verify Dataset

> Run this after Section 2 (or after restoring from Drive) to confirm counts.


In [ ]:
import json, re
from pathlib import Path

DATASET = '/content/dataset_v6'

# Restore from Drive if needed (e.g., after session disconnect)
import os, zipfile
ZIP_PATH = '/content/drive/MyDrive/DegreeDetailExtract/dataset_v6.zip'
if not os.path.exists(DATASET) and os.path.exists(ZIP_PATH):
    print('Restoring dataset from Drive...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done.')

for split in ['train', 'val', 'test']:
    meta = Path(DATASET) / f'metadata_{split}.jsonl'
    if meta.exists():
        n = sum(1 for _ in open(meta))
        print(f'  {split}: {n} samples')

# Check date normalization
sample = json.loads(open(Path(DATASET) / 'metadata.jsonl').readline())
gt = sample.get('ground_truth', '')
dm = re.search(r'<s_issue_date>(.*?)</s_issue_date>', gt)
print(f'Sample GT date (expected DD-MM-YYYY): {dm.group(1) if dm else "N/A"}')


## Section 4 — Load Model & Processor

> Run every session. The base model downloads once to Colab disk (~800MB).
> Takes ~3-4 min first time; ~1 min with cache.


In [ ]:
import os, sys, json, re, torch
from pathlib import Path
from transformers import DonutProcessor, VisionEncoderDecoderModel

DATASET        = '/content/dataset_v6'
MODEL_BASE     = 'naver-clova-ix/donut-base'
CHECKPOINT_DIR = '/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v6'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

FIELDS = [
    'student_name', 'university_name', 'course_name',
    'specialization', 'pass_class', 'authority_name', 'issue_date'
]
TASK_START_TOKEN = '<s_cert>'
TASK_END_TOKEN   = '</s_cert>'
SPECIAL_TOKENS = (
    [TASK_START_TOKEN, TASK_END_TOKEN]
    + [f'<s_{f}>' for f in FIELDS]
    + [f'</s_{f}>' for f in FIELDS]
)

# Load processor
processor = DonutProcessor.from_pretrained(MODEL_BASE)
processor.tokenizer.add_special_tokens({'additional_special_tokens': SPECIAL_TOKENS})
processor.image_processor.size = {'height': 960, 'width': 720}

print(f'Vocab size: {len(processor.tokenizer)}')
print(f'Image size: {processor.image_processor.size}')


In [ ]:
# ── Check if best checkpoint exists on Drive to continue from ──────────
BEST_CKPT  = f'{CHECKPOINT_DIR}/best_model'
STATE_PATH = f'{CHECKPOINT_DIR}/train_state_v6.pt'

if os.path.exists(BEST_CKPT):
    print(f'Found existing checkpoint: {BEST_CKPT}')
    print('Will load weights from checkpoint instead of base model.')
    model_path = BEST_CKPT
else:
    print('No checkpoint found — loading base Donut model (first run).')
    model_path = MODEL_BASE

model = VisionEncoderDecoderModel.from_pretrained(model_path)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.pad_token_id  = processor.tokenizer.pad_token_id
model.config.eos_token_id  = processor.tokenizer.eos_token_id
model.config.decoder_start_token_id = (
    processor.tokenizer.convert_tokens_to_ids([TASK_START_TOKEN])[0]
)

# Freeze encoder initially — unfreeze in final 3 epochs to fine-tune vision
for param in model.encoder.parameters():
    param.requires_grad = False

model = model.to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total: {total/1e6:.1f}M params | Trainable (decoder): {trainable/1e6:.1f}M')


## Section 5 — Dataset Class & DataLoaders

In [ ]:
import json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

# ── Free tier recommended batch size ─────────────────────────────────────
# T4 (15GB): batch_size=2 + grad_accum=8 → effective batch=16. Safe.
# If you get CUDA OOM, drop batch_size to 1.
BATCH_SIZE    = 2
GRAD_ACCUM    = 8   # Effective batch = BATCH_SIZE * GRAD_ACCUM = 16
MAX_LABEL_LEN = 256  # Increased from v5's 128 to prevent tag truncation

# ── On-the-fly dynamic augmentation (CRITICAL for fighting overfitting) ────
# This ensures the model never sees the exact same image pixels twice across epochs.
# 1,920 images × 12 epochs = 23,040 uniquely perturbed views!
train_img_transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomRotation(degrees=(-2, 2), fill=255),
    transforms.RandomPerspective(distortion_scale=0.03, p=0.25, fill=255),
])

class CertDatasetV6(Dataset):
    def __init__(self, jsonl_path, processor, img_base_dir, max_length=256, is_train=False):
        self.records = []
        with open(jsonl_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.records.append(json.loads(line))
        self.processor  = processor
        self.img_base   = Path(img_base_dir)
        self.max_length = max_length
        self.is_train   = is_train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img_path = self.img_base / rec['file_name']
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (720, 960), (255, 255, 255))

        # Dynamic augmentation during training (keeps text readable while varying pixels)
        if self.is_train:
            img = train_img_transform(img)

        pixel_values = self.processor(
            images=img, return_tensors='pt'
        ).pixel_values.squeeze(0)

        gt = rec.get('ground_truth', '') + self.processor.tokenizer.eos_token
        labels = self.processor.tokenizer(
            gt,
            add_special_tokens=False,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


train_ds = CertDatasetV6(f'{DATASET}/metadata_train.jsonl', processor, DATASET, is_train=True)
val_ds   = CertDatasetV6(f'{DATASET}/metadata_val.jsonl',   processor, DATASET, is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=0)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
print(f'Batches/epoch: {len(train_loader)} | Grad accum: {GRAD_ACCUM}')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')

# ── Rough time estimate ───────────────────────────────────────────────────
# T4 processes ~2 batches/sec → epoch time ≈ len(train_loader)/2 seconds
import math
approx_min = len(train_loader) / 2 / 60
print(f'Approx train time per epoch: ~{approx_min:.0f} min on T4')
print(f'Val time per epoch (greedy): ~{len(val_ds)*0.5/60:.0f} min on T4')


## Section 6 — Evaluation Metrics

In [ ]:
import re

def strip_xml(text):
    return re.sub(r'<[^>]+>', '', text).strip()

def parse_fields_from_xml(xml_str):
    result = {}
    for f in FIELDS:
        m = re.search(rf'<s_{f}>(.*?)</s_{f}>', xml_str, re.DOTALL)
        result[f] = strip_xml(m.group(1)) if m else ''
    return result

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    dp = list(range(len(b)+1))
    for ca in a:
        new_dp = [dp[0]+1]
        for j, cb in enumerate(b):
            new_dp.append(min(dp[j+1]+1, new_dp[j]+1, dp[j]+(0 if ca==cb else 1)))
        dp = new_dp
    return dp[-1]

def compute_field_metrics(preds, gts):
    metrics = {f: {'exact': 0, 'cer_sum': 0.0, 'n': 0} for f in FIELDS}
    for pred, gt in zip(preds, gts):
        for f in FIELDS:
            pv = pred.get(f,'').lower().strip()
            gv = gt.get(f,'').lower().strip()
            if not gv: continue
            metrics[f]['n'] += 1
            if pv == gv: metrics[f]['exact'] += 1
            metrics[f]['cer_sum'] += levenshtein(pv, gv) / max(len(gv), 1)
    rows = []
    for f in FIELDS:
        m = metrics[f]; n = m['n']
        rows.append({
            'field': f,
            'n': n,
            'exact%': 100*m['exact']/n if n else 0,
            'CER': m['cer_sum']/n if n else 0
        })
    return rows

print('Metric functions ready.')


## Section 7 — Training Loop

### Free Tier Strategy:
- **Target: 15 epochs total** across 1-2 sessions (~2-3h per session on T4)
- **Encoder unfreezes at epoch 12** (last 3 epochs fine-tune full model)
- **Greedy decoding during training** — much faster than beam search
- **Beam search (num_beams=4) only in Section 8** (final evaluation)
- **Early stopping patience=3** — stops if no improvement for 3 epochs
- **Every epoch saves to Drive** — safe to disconnect and resume

> **To resume after disconnect:** Re-run Sections 0 → 4 → 5 → 6 → then this section.
> The checkpoint state on Drive is loaded automatically.


In [ ]:
import math, torch, torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm

# ── Hyperparameters ──────────────────────────────────────────────────────
# Free tier optimized — calibrated against overfitting and runtime limits
NUM_EPOCHS      = 12         # 12 epochs with dynamic augmentation prevents overfit
UNFREEZE_EPOCH  = 9          # Decoder-only for epochs 1-8, unfreeze encoder for last 3
LR_DECODER      = 5e-5       # Phase 1: decoder-only
LR_FULL         = 1e-5       # Phase 2: full model (gentler LR)
GRAD_ACCUM      = 8          # Effective batch = 16
LABEL_SMOOTHING = 0.1        # Regularization against overconfidence
PATIENCE        = 3          # Stop early if val_loss doesn't improve for 3 epochs
MAX_NEW_TOKENS  = 256        # v6: increased from 128 to prevent truncated tags
# NOTE: During training we use greedy decode (fast).
# Beam search (num_beams=4) is only used in Section 8 for final eval.

print('Config:')
print(f'  Epochs: {NUM_EPOCHS} | Unfreeze at: {UNFREEZE_EPOCH}')
print(f'  LR: {LR_DECODER} -> {LR_FULL} | Grad accum: {GRAD_ACCUM}')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Label smoothing: {LABEL_SMOOTHING} | Patience: {PATIENCE}')


In [ ]:
# ── Label smoothing loss ─────────────────────────────────────────────────
def smooth_cross_entropy(logits, labels, smoothing=0.1):
    vocab = logits.size(-1)
    logits_flat = logits.view(-1, vocab)
    labels_flat = labels.view(-1)
    mask = labels_flat != -100
    if not mask.any():
        return torch.tensor(0.0, device=logits.device)
    logits_flat = logits_flat[mask]
    labels_flat = labels_flat[mask]
    log_probs = F.log_softmax(logits_flat, dim=-1)
    nll  = F.nll_loss(log_probs, labels_flat, reduction='mean')
    smooth = -log_probs.mean()
    return (1.0 - smoothing) * nll + smoothing * smooth

# ── Optimizer & Scheduler ────────────────────────────────────────────────
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_DECODER, weight_decay=0.01
)
total_steps = NUM_EPOCHS * len(train_loader) // GRAD_ACCUM
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-7)

# ── Resume from checkpoint ────────────────────────────────────────────────
STATE_PATH = f'{CHECKPOINT_DIR}/train_state_v6.pt'
start_epoch    = 0
best_val_loss  = float('inf')
patience_count = 0
encoder_frozen = True
history = {'train_loss': [], 'val_loss': [], 'field_metrics': []}

if os.path.exists(STATE_PATH):
    print('Loading checkpoint from Drive...')
    state = torch.load(STATE_PATH, map_location=DEVICE)
    model.load_state_dict(state['model'])
    optimizer.load_state_dict(state['optimizer'])
    scheduler.load_state_dict(state['scheduler'])
    start_epoch    = state['epoch'] + 1
    best_val_loss  = state['best_val_loss']
    patience_count = state.get('patience_count', 0)
    history        = state.get('history', history)
    encoder_frozen = state.get('encoder_frozen', True)
    if not encoder_frozen:
        for param in model.encoder.parameters():
            param.requires_grad = True
    print(f'  Resumed from epoch {start_epoch}')
    print(f'  Best val_loss so far: {best_val_loss:.4f}')
    print(f'  Patience: {patience_count}/{PATIENCE}')
else:
    print('No checkpoint found — starting from epoch 0.')


In [ ]:
# ── Enable gradient checkpointing (saves ~4GB VRAM on T4) ───────────────
model.gradient_checkpointing_enable()

print('Starting training...')
print('Checkpoint saved to Drive after each epoch — safe to disconnect.')
print('-' * 60)

for epoch in range(start_epoch, NUM_EPOCHS):

    # ── Unfreeze encoder for last 3 epochs ───────────────────────────────
    if epoch == UNFREEZE_EPOCH and encoder_frozen:
        print(f'\n[Epoch {epoch+1}] Unfreezing encoder — full fine-tuning starts.')
        for param in model.encoder.parameters():
            param.requires_grad = True
        encoder_frozen = False
        for pg in optimizer.param_groups:
            pg['lr'] = LR_FULL
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable params: {trainable/1e6:.1f}M')

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    train_loss_sum = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f'Ep {epoch+1}/{NUM_EPOCHS} train', leave=False)
    for step, batch in enumerate(pbar):
        pv = batch['pixel_values'].to(DEVICE)
        lb = batch['labels'].to(DEVICE)
        out = model(pixel_values=pv, labels=lb)
        loss = smooth_cross_entropy(out.logits, lb, LABEL_SMOOTHING) / GRAD_ACCUM
        loss.backward()
        train_loss_sum += loss.item() * GRAD_ACCUM
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix({'loss': f'{loss.item()*GRAD_ACCUM:.4f}',
                              'lr':   f'{scheduler.get_last_lr()[0]:.1e}'})

    avg_train = train_loss_sum / len(train_loader)
    history['train_loss'].append(avg_train)

    # ── Validate ──────────────────────────────────────────────────────────
    # GREEDY decoding during training (fast — ~30s per 300 val samples)
    # Beam search only in Section 8 final eval
    model.eval()
    val_loss_sum = 0.0
    val_preds, val_gts = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Ep {epoch+1}/{NUM_EPOCHS} val', leave=False):
            pv = batch['pixel_values'].to(DEVICE)
            lb = batch['labels'].to(DEVICE)

            # Loss
            out = model(pixel_values=pv, labels=lb)
            val_loss_sum += smooth_cross_entropy(out.logits, lb, LABEL_SMOOTHING).item()

            # Greedy generate
            did = torch.full((1,1), model.config.decoder_start_token_id,
                             dtype=torch.long, device=DEVICE)
            gen = model.generate(
                pixel_values=pv,
                decoder_input_ids=did,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=1,      # Greedy: fast during training
                early_stopping=True
            )
            pred_str = processor.tokenizer.decode(gen[0], skip_special_tokens=False)
            val_preds.append(parse_fields_from_xml(pred_str))

            gt_lb = lb.clone()
            gt_lb[gt_lb == -100] = processor.tokenizer.pad_token_id
            gt_str = processor.tokenizer.decode(gt_lb[0], skip_special_tokens=True)
            val_gts.append(parse_fields_from_xml(gt_str))

    avg_val = val_loss_sum / len(val_loader)
    history['val_loss'].append(avg_val)
    field_rows = compute_field_metrics(val_preds, val_gts)
    history['field_metrics'].append(field_rows)

    # ── Print results ─────────────────────────────────────────────────────
    print(f'\nEpoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'train={avg_train:.4f} | val={avg_val:.4f} | '
          f'lr={scheduler.get_last_lr()[0]:.1e}')
    print(f'  {"Field":<22} {"Exact%":>8} {"CER":>8}')
    for row in field_rows:
        print(f'  {row["field"]:<22} {row["exact%"]:>7.1f}% {row["CER"]:>8.3f}')

    # ── Save best checkpoint ──────────────────────────────────────────────
    if avg_val < best_val_loss:
        best_val_loss  = avg_val
        patience_count = 0
        best_dir = f'{CHECKPOINT_DIR}/best_model'
        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)
        print(f'  >>> NEW BEST val={best_val_loss:.4f} — saved to Drive')
    else:
        patience_count += 1
        print(f'  Patience: {patience_count}/{PATIENCE}')

    # ── Save full training state (for resume across sessions) ─────────────
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
        'patience_count': patience_count,
        'history': history,
        'encoder_frozen': encoder_frozen,
    }, STATE_PATH)

    # ── Early stopping ────────────────────────────────────────────────────
    if patience_count >= PATIENCE:
        print(f'\n>>> Early stopping at epoch {epoch+1}. Best val={best_val_loss:.4f}')
        break

print('\n=== Training complete ===')
print(f'Best val_loss: {best_val_loss:.4f}')


## Section 8 — Training Curves

> Run anytime after at least one epoch completes.


In [ ]:
import matplotlib.pyplot as plt

if not history['train_loss']:
    print('No training history yet. Run Section 7 first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.plot(history['train_loss'], 'o-', label='Train Loss', ms=4)
    ax.plot(history['val_loss'],   's-', label='Val Loss',   ms=4)
    ax.set_title('v6 Loss Curves', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    ax2 = axes[1]
    if history['field_metrics']:
        last = history['field_metrics'][-1]
        fnames    = [r['field'] for r in last]
        exact_pct = [r['exact%'] for r in last]
        bars = ax2.bar(fnames, exact_pct, color='steelblue', edgecolor='white')
        ax2.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
        ax2.set_title('Per-Field Exact Match % (Last Epoch)', fontsize=13)
        ax2.set_ylabel('Exact Match %'); ax2.set_ylim(0, 105)
        ax2.tick_params(axis='x', rotation=30); ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{CHECKPOINT_DIR}/v6_training_curves.png', dpi=120)
    plt.show()
    print('Saved to Drive.')


## Section 9 — Final Evaluation on Test Set

> Run **after training completes** (or after loading best model from Drive).
> This is the only place we use **beam search (num_beams=4)** for best accuracy.
> Expect ~5-10 min for the full test set.


In [ ]:
# Load best model for evaluation
from transformers import DonutProcessor, VisionEncoderDecoderModel
from torch.utils.data import DataLoader

best_dir = f'{CHECKPOINT_DIR}/best_model'
if not os.path.exists(best_dir):
    print('ERROR: No best model found. Train first (Section 7).')
else:
    proc_eval  = DonutProcessor.from_pretrained(best_dir)
    model_eval = VisionEncoderDecoderModel.from_pretrained(best_dir).to(DEVICE)
    model_eval.eval()
    print('Best model loaded.')

    test_ds = CertDatasetV6(
        f'{DATASET}/metadata_test.jsonl', proc_eval, DATASET, max_length=256
    )
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)
    print(f'Test samples: {len(test_ds)}')


In [ ]:
test_preds, test_gts = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test Eval'):
        pv = batch['pixel_values'].to(DEVICE)
        lb = batch['labels'].to(DEVICE)

        did = torch.full((1,1), model_eval.config.decoder_start_token_id,
                         dtype=torch.long, device=DEVICE)
        gen = model_eval.generate(
            pixel_values=pv,
            decoder_input_ids=did,
            max_new_tokens=256,
            num_beams=4,          # Beam search for best accuracy
            early_stopping=True
        )
        pred_str = proc_eval.tokenizer.decode(gen[0], skip_special_tokens=False)
        test_preds.append(parse_fields_from_xml(pred_str))

        gt_lb = lb.clone()
        gt_lb[gt_lb == -100] = proc_eval.tokenizer.pad_token_id
        gt_str = proc_eval.tokenizer.decode(gt_lb[0], skip_special_tokens=True)
        test_gts.append(parse_fields_from_xml(gt_str))

# Print results
metrics = compute_field_metrics(test_preds, test_gts)
print('\n=== v6 Test Set Results (beam_search=4) ===')
print(f'{"Field":<22} {"N":>5} {"Exact%":>9} {"CER":>8}')
print('=' * 48)
for row in metrics:
    print(f'{row["field"]:<22} {row["n"]:>5} {row["exact%"]:>8.1f}% {row["CER"]:>8.3f}')
avg_exact = sum(r['exact%'] for r in metrics) / len(metrics)
avg_cer   = sum(r['CER'] for r in metrics) / len(metrics)
print('=' * 48)
print(f'{"Average":<22} {"":>5} {avg_exact:>8.1f}% {avg_cer:>8.3f}')


## Section 10 — Deploy / Use Model

The best checkpoint is at:
`/content/drive/MyDrive/DegreeDetailExtract/checkpoints_v6/best_model`

Download it to your local machine and place at:
`v:\Vrushti\Projects\DegreeDetailExtract\checkpoints\best_model`

Then start the v6 backend:
```powershell
uvicorn main_v6:app --reload --port 8000
```


In [ ]:
# Optional: push to HuggingFace Hub so you can load the model from anywhere
# from huggingface_hub import notebook_login
# notebook_login()
# model_eval.push_to_hub('your-hf-username/degree-extract-v6')
# proc_eval.push_to_hub('your-hf-username/degree-extract-v6')

print('Best model path on Drive:')
print(f'  {CHECKPOINT_DIR}/best_model')
print('Download to local: checkpoints/best_model')
print('Backend command:  uvicorn main_v6:app --reload --port 8000')
